In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_df = spark.table(
    "banking_catalog.banking_schema.bronze_transactions"
)
display(bronze_df.limit(5))

timestamp,from_bank,account2,to_bank,account4,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
2022/09/04 04:45,021940,8022BFE30,011474,8034A4730,45.25,US Dollar,45.25,US Dollar,Cash,0
2022/09/04 04:43,021940,8022BFE30,011474,8034A4730,1.67,US Dollar,1.67,US Dollar,ACH,0
2022/09/04 04:54,011471,800C26050,01674,8034D8070,438714.34,US Dollar,438714.34,US Dollar,Cheque,0
2022/09/04 04:30,021575,8032DF620,023842,803510C50,2440.53,US Dollar,2440.53,US Dollar,Cheque,0
2022/09/04 04:38,021575,8032DF620,023842,803510C50,699.42,US Dollar,699.42,US Dollar,Credit Card,0


In [0]:
bronze_df.count()

5078345

In [0]:
bronze_df.groupBy(bronze_df.columns)\
    .count()\
    .filter("count > 1")\
    .count()

9

In [0]:
from pyspark.sql.functions import col, sum, when

bronze_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in bronze_df.columns
]).show()

+---------+---------+--------+-------+--------+---------------+------------------+-----------+----------------+--------------+-------------+
|timestamp|from_bank|account2|to_bank|account4|amount_received|receiving_currency|amount_paid|payment_currency|payment_format|is_laundering|
+---------+---------+--------+-------+--------+---------------+------------------+-----------+----------------+--------------+-------------+
|        0|        0|       0|      0|       0|              0|                 0|          0|               0|             0|            0|
+---------+---------+--------+-------+--------+---------------+------------------+-----------+----------------+--------------+-------------+



In [0]:
bronze_df.select("timestamp").show(10, False)

+----------------+
|timestamp       |
+----------------+
|2022/09/04 04:45|
|2022/09/04 04:43|
|2022/09/04 04:54|
|2022/09/04 04:30|
|2022/09/04 04:38|
|2022/09/04 04:59|
|2022/09/04 04:41|
|2022/09/04 04:53|
|2022/09/04 04:53|
|2022/09/04 04:42|
+----------------+
only showing top 10 rows


In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed("account2", "sender_account")
    .withColumnRenamed("account4", "receiver_account")
    .withColumn(
        "timestamp",
        to_timestamp(col("timestamp"), "yyyy/MM/dd HH:mm")
    )
    .withColumn(
        "amount_received",
        col("amount_received").cast(DecimalType(18,2))
    )
    .withColumn(
        "amount_paid",
        col("amount_paid").cast(DecimalType(18,2))
    )
    .withColumn(
        "is_laundering",
        col("is_laundering").cast("int")
    )
    .dropDuplicates()
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("load_date", current_date())
)

In [0]:
silver_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- from_bank: string (nullable = true)
 |-- sender_account: string (nullable = true)
 |-- to_bank: string (nullable = true)
 |-- receiver_account: string (nullable = true)
 |-- amount_received: decimal(18,2) (nullable = true)
 |-- receiving_currency: string (nullable = true)
 |-- amount_paid: decimal(18,2) (nullable = true)
 |-- payment_currency: string (nullable = true)
 |-- payment_format: string (nullable = true)
 |-- is_laundering: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- load_date: date (nullable = false)



In [0]:
silver_df.count()

5077947

In [0]:
silver_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable(
        "banking_catalog.banking_schema.silver_transactions"
    )

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM banking_catalog.banking_schema.silver_transactions
""").show()

+--------+
|COUNT(*)|
+--------+
| 5077947|
+--------+

